### Agent

<small>User prompt: "Schedule a meeting with Joe next week and send him a confirmation email."

To complete this, the LLM with help of Agent will do the following tasks:  
- Check your calendar
- Find free time slots
- Check Joe’s availability
- Create the calendar event
- Send a confirmation email
- Ask follow-up questions if information is missing</small>

**Agent Loop**  
<small>It repeatedly performs:  
- Reason  
- Choose an action  
- Call a tool  
- Observe result  
- Update plan  
- Repeat until done</nsmall>

### ReAct pattern(Reason + Act)

```mermaid
---
title: ReAct Pattern
---
flowchart LR
    A[User Query] --> B(REason \n Analyze & Decide)
    B --> C>Task Complete?]
    C -->|Yes| D[Final Result]
    C -->|No| E[ACTion \nUse Tools & Execute]
    E --> F[Observation \n review result]
    F --> B
```

<small>ReAct pattern combines:  
- Reasoning steps (“Thought”)  
- Actions (“Tool Calls”)  
- Observations (“Tool Results”)  

**Typical Flow**:  

Thought: I need weather data  
Action: call_weather_api("Bangalore")  
Observation: Rainy, 24°C  

Thought: Need calendar availability  
Action: call_calendar_api()  
Observation: Free at 4 PM  

Thought: Enough information gathered  
Final Answer: Schedule outdoor meeting tomorrow instead  

The model alternates between:  
Internal reasoning  
External actions</nsmall>

### Tools

<small>Tools are external functions/APIs the agent can use.  
Examples:  
Weather API  
Calendar API   
Search engine   
Email sender</nsmall>

<small>How LLM selects the appropriate tool?  
The LLM reads:   
- User request  
- Available tool descriptions  
- Previous observations  

Then predicts: Which tool best helps accomplish the task?</nsmall>

<small>Available Tools
| Tool           | Purpose               |
| -------------- | --------------------- |
| `search_web`   | Search internet       |
| `send_email`   | Send email            |
| `create_event` | Create calendar event |</small>


JSON Tool Schema

In [ ]:
{
  "name": "send_email",
  "description": "Send an email to a recipient",
  "parameters": {
    "type": "object",
    "properties": {
      "to": {
        "type": "string"
      },
      "subject": {
        "type": "string"
      },
      "body": {
        "type": "string"
      }
    },
    "required": ["to", "subject", "body"]
  }
}

<small> Using the available tools and their tool schemas the model decides the tools that are required and passes the appropriate arguments.  
send_email(  
&emsp;to="joe@example.com",  
&emsp;subject="Meeting Confirmed",  
&emsp;body="Your meeting is confirmed."  
)</small>

Tool creation and management

In [1]:
from abc import ABC, abstractmethod
import requests
import os

class Tool(ABC):
    name = ""
    description = ""
    @abstractmethod
    def run(self, **kwargs):
        pass

In [2]:
class WeatherTool(Tool):
    name = "weather"
    description = (
        "Get current weather information for a city."
    )
    def __init__(self, api_key):
        self.api_key = api_key
    def run(self, city):
        url = "https://api.openweathermap.org/data/2.5/weather"
        params = {
            "q": city,
            "appid": self.api_key,
            "units": "metric"
        }
        response = requests.get(url, params=params)
        if response.status_code != 200:
            return f"Error: {response.json().get('message')}"
        data = response.json()
        weather = data["weather"][0]["description"]
        temp = data["main"]["temp"]
        humidity = data["main"]["humidity"]
        return (
            f"Weather in {city}:\n"
            f"Temperature: {temp}°C\n"
            f"Condition: {weather}\n"
            f"Humidity: {humidity}%"
        )

In [ ]:
class ExchangeRateTool(Tool):
    name = "exchange_rate"
    description = (
        "Get exchange rate between two currencies."
    )
    def __init__(self, api_key):
        self.api_key = api_key
    def run(self, base_currency, target_currency):
        url = (
            f"https://v6.exchangerate-api.com/v6/"
            f"{self.api_key}/latest/{base_currency.upper()}"
        )
        response = requests.get(url)
        if response.status_code != 200:
            return "Failed to fetch exchange rates."
        data = response.json()
        if data["result"] != "success":
            return data.get("error-type", "Unknown error")
        rates = data["conversion_rates"]
        if target_currency.upper() not in rates:
            return f"Currency '{target_currency}' not found."
        rate = rates[target_currency.upper()]
        return (
            f"Exchange Rate:\n"
            f"1 {base_currency.upper()} = "
            f"{rate} {target_currency.upper()}"
        )

In [ ]:
class NewsTool(Tool):
    name = "news"
    description = (
        "Get latest news headlines for a topic."
    )
    def __init__(self, api_key):
        self.api_key = api_key
    def run(self, topic="technology", limit=5):
        url = "https://newsapi.org/v2/everything"
        params = {
            "q": topic,
            "sortBy": "publishedAt",
            "language": "en",
            "pageSize": limit,
            "apiKey": self.api_key
        }
        response = requests.get(url, params=params)
        if response.status_code != 200:
            return "Failed to fetch news."
        data = response.json()
        if data["status"] != "ok":
            return data.get("message", "Unknown error")
        articles = data["articles"]
        if not articles:
            return f"No news found for '{topic}'."
        result = [f"Latest news about {topic}:\n"]
        for i, article in enumerate(articles, start=1):
            result.append(
                f"{i}. {article['title']}\n"
                f"   Source: {article['source']['name']}\n"
                f"   URL: {article['url']}\n"
            )
        return "\n".join(result)

In [ ]:
class TimezoneTool(Tool):
    name = "timezone"
    description = (
        "Get timezone information from a timezone name."
    )
    def __init__(self, api_key):
        self.api_key = api_key
    def run(self, timezone):
        url = "https://api.api-ninjas.com/v1/timezone"
        headers = {
            "X-Api-Key": self.api_key
        }
        params = {
            "timezone": timezone
        }
        response = requests.get(
            url,
            headers=headers,
            params=params
        )
        if response.status_code != 200:
            return f"Error: {response.text}"
        data = response.json()
        return (
            f"Timezone: {data['timezone']}\n"
            f"Local Time: {data['local_time']}\n"
            f"UTC Offset: {data['utc_offset']} seconds"
        )

In [ ]:
class EmailTool(Tool):
    name = "email"
    description = "Send an email using Resend."
    def __init__(self, api_key):
        self.api_key = api_key
    def run(self, to, subject, body):
        url = "https://api.resend.com/emails"
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json"
        }
        payload = {
            "from": "onboarding@resend.dev",
            "to": [to],
            "subject": subject,
            "html": f"<p>{body}</p>"
        }
        response = requests.post(
            url,
            headers=headers,
            json=payload
        )
        if response.status_code not in [200, 201]:
            return f"Error: {response.text}"
        data = response.json()
        return (
            "Email sent successfully!\n"
            f"Email ID: {data.get('id')}"
        )

In [33]:
# Load API key
WEATHER_API_KEY = os.getenv("OPENWEATHER_API_KEY")
EXCHANGE_API_KEY = os.getenv("EXCHANGERATE_API_KEY")
NEWS_API_KEY = os.getenv("NEWSAPI_API_KEY")
NINJAS_API_KEY = os.getenv("APININJAS_API_KEY")
RESEND_API_KEY = os.getenv("RESEND_API_KEY")
# Tools registry
TOOLS = {
    "weather": WeatherTool(WEATHER_API_KEY),
    "exchange_rate": ExchangeRateTool(EXCHANGE_API_KEY),
    "news": NewsTool(NEWS_API_KEY),
    "timezone": TimezoneTool(NINJAS_API_KEY),
    "email": EmailTool(RESEND_API_KEY)
}

In [8]:
def execute_tool(tool_name, **kwargs):
    tool = TOOLS.get(tool_name)
    if not tool:
        return "Tool not found"
    return tool.run(**kwargs)

In [9]:
result = execute_tool(
    "weather",
    city="Bangalore"
)
print(result)

Weather in Bangalore:
Temperature: 29.07°C
Condition: overcast clouds
Humidity: 60%


In [10]:
result = execute_tool(
    "exchange_rate",
    base_currency="USD",
    target_currency="INR"
)
print(result)

Exchange Rate:
1 USD = 95.7911 INR


In [13]:
result = execute_tool(
    "news",
    topic="Stocks",
    limit=3
)
print(result)

Latest news about Stocks:

1. Breakout #2—100 Posters by Cihan Tamti Proves That Daily Practice Builds a Design Language
   Source: Weandthecolor.com
   URL: https://weandthecolor.com/breakout-2-100-posters-by-cihan-tamti-proves-that-daily-practice-builds-a-design-language/210116

2. Nifty Bank tumbles 600 points; IndusInd Bank, Yes Bank, SBI and other stocks fall up to 3%. What lies ahead?
   Source: The Times of India
   URL: https://economictimes.indiatimes.com/markets/stocks/news/nifty-bank-tumbles-600-points-indusind-bank-yes-bank-sbi-and-other-stocks-fall-up-to-3-what-lies-ahead/articleshow/131478355.cms

3. Bitcoin falls 12% in a week, drops below key $70K level as ETF outflows accelerate
   Source: The Times of India
   URL: https://economictimes.indiatimes.com/markets/cryptocurrency/crypto-news/bitcoin-falls-12-in-a-week-drops-below-key-70k-level-as-etf-outflows-accelerate/articleshow/131478285.cms



In [35]:
result = execute_tool(
    "timezone",
    timezone="Europe/London"
)

print(result)

Timezone: Europe/London
Local Time: 2026-06-04 08:20:14
UTC Offset: 3600 seconds


In [21]:
result = execute_tool(
    "email",
    to="ogiboyinaakash@gmail.com",
    subject="Hello",
    body="This email was sent by my AI agent."
)
print(result)

Email sent successfully!
Email ID: cd8aadaa-fb57-4d2a-8ca9-0896509d08c4


### Multi-Step Task Execution

<small>Example task:  
"Book a flight and add it to my calendar."  

The agent may need:  
- Search flights  
- Compare prices  
- Choose best option  
- Book ticket  
- Generate confirmation  
- Add calendar event  
- Notify user  

This requires:  
- State tracking  
- Iterative planning  
- Tool orchestration  
- Failure handling </nsmall>

<small>**Planning Loop**  
while not task_completed:  
&emsp;think()  
&emsp;choose_action()  
&emsp;execute_tool()  
&emsp;observe_result()  
&emsp;update_plan()</nsmall>

#### Loop Control Mechanisms  

<small>Without loop control, agents can:   
- Run forever  
- Repeat useless actions  
- Spam APIs   
- Waste tokens  
- Enter failure cycles </nsmall>

**1. Max Iterations**  
<small>Limits total reasoning cycles.  

Example:  
MAX_ITERATIONS = k  
while step_count < MAX_STEPS:
    run_agent_step()
if step_count > MAX_ITERATIONS:  
&emsp;terminate("Task stopped: iteration limit exceeded")  

Purpose:  
- Prevent infinite loops  
- Control cost  
- Ensure predictable runtime</nsmall>

**2. Stopping Conditions**  
<small>Agent should stop when:  
- Goal achieved  
- Tool says “done”  
- Confidence is high  
- No more actions available  
- Repeated failures detected  

Example:   
if state["booking_confirmed"]:  
&emsp;break</nsmall>

### Error Recovery

<small>most beginner implementations:  
try:  
&emsp;tool.run()  
except:  
&emsp;pass  
This is NOT sufficient. 
 
Real agents must:  
- Diagnose failures   
- Retry intelligently  
- Switch strategies  
- Use fallback tools  
- Ask clarifying questions  
- Modify plans dynamically</nsmall>

<small>Types of tool failures
| Failure Type            | Example            |
| ----------------------- | ------------------ |
| Timeout                 | API takes too long |
| Invalid output          | Missing fields     |
| Rate limit              | Too many requests  |
| Tool unavailable        | Service down       |
| Hallucinated tool usage | Wrong parameters   |
| Parsing failure         | JSON malformed     |
| Empty result            | No search results  |
| Authentication error    | Invalid API key    |</nsmall>

<small>**Deliberate Failure**  
User asks: "Schedule a meeting tomorrow at 4 PM."    
Agent uses:  
- Calendar tool  
- Notification tool  

Calendar tool fails with:  
{  
&emsp;"error": "503 Service Unavailable"  
}  
A weak agent crashes.  
A strong agent recovers.</nsmall>

#### Recovery Logic

<small>**Detect Failure**  
result = calendar_tool.run(data)  
if "error" in result:  
&emsp;tool_failed = True</nsmall>

<small>**Failure classification**  
error_type = classify_error(result) 

Classifications:   
- TEMPORARY  
- INVALID_INPUT  
- AUTH_FAILURE  
- EMPTY_RESULT
- UNKNOWN</nsmall>

<small>**Recovery Strategies**  
| Error Type       | Recovery            |
| ---------------- | ------------------- |
| Timeout          | Retry               |
| Rate limit       | Backoff and retry   |
| Invalid input    | Reformat parameters |
| Empty result     | Alternative query   |
| Tool unavailable | Use fallback tool   |
| Auth error       | Escalate to user    |
| Unknown          | Abort safely        |</nsmall>

<small>**Retry Logic**  
Do not just implement: try again  

Instead do: 
- Limit retries  
- Change behavior  
- Add delay  
- Modify parameters  

Example: **exponential backoff**  
retry_count += 1  
if retry_count <= MAX_RETRIES:  
&emsp;wait(2 ** retry_count)</nsmall>

<small>**Fallback Tool Strategy**  
If primary calendar API fails:  
GoogleCalendarTool → OutlookCalendarTool  

Example: **real agent resilience**  
if google_failed:  
&emsp;use_outlook_calendar()

<small>**Re-Planning After Failure**   
The best agents re-plan dynamically. 

Example:  
Thought: Calendar API unavailable.  
Alternative approach: Store reminder locally and notify user.</nsmall>

<small>**Full Recovery Loop Example**

MAX_ITERATIONS = 5  
MAX_RETRIES = 2  

for iteration in range(MAX_ITERATIONS):  
&emsp;thought = agent.reason(state)  
&emsp;action = agent.choose_action(thought)  
&emsp;tool = tools[action["tool"]]  
&emsp;result = tool.run(action["input"])  
&emsp;# Failure handling  
&emsp;if "error" in result:  
&emsp;&emsp;error_type = classify_error(result)  
&emsp;&emsp;if error_type == "TEMPORARY":  
&emsp;&emsp;&emsp;retries += 1  
&emsp;&emsp;&emsp;if retries <= MAX_RETRIES:  
&emsp;&emsp;&emsp;&emsp;sleep(2 ** retries)    
&emsp;&emsp;&emsp;&emsp;continue  
&emsp;&emsp;elif error_type == "INVALID_INPUT":  
&emsp;&emsp;&emsp;action["input"] = repair_input(action["input"])   
&emsp;&emsp;&emsp;continue  
&emsp;&emsp;elif error_type == "TOOL_UNAVAILABLE":   
&emsp;&emsp;&emsp;tool = fallback_tools[action["tool"]]  
&emsp;&emsp;&emsp;result = tool.run(action["input"])  
&emsp;&emsp;else:  
&emsp;&emsp;&emsp;return "Task failed safely"  
&emsp;state.update(result)  
&emsp;if state["goal_completed"]:  
&emsp;&emsp;break   
</nsmall>

<small>**Agentic Recovery**  
detect_failure()  
classify_failure()  
choose_recovery_strategy()  
retry_or_replan()  
continue_execution()  

**Agent Structure**  
state = {  
&emsp;"goal": "...",  
&emsp;"completed_steps": [],  
&emsp;"failed_steps": [],  
&emsp;"tool_history": [],   
&emsp;"retry_counts": {},  
&emsp;"observations": [],  
&emsp;"current_plan": []  
}

**Structured Tool Output**  
{  
&emsp;"status": "success",  
&emsp;"data": {...}  
}  

**Tool History Tracking**  
tool_history.append({  
&emsp;"tool": "calendar",  
&emsp;"input": data,  
&emsp;"result": result  
})</nsmall>

### Agent Safety 

**Permission boundaries :**  
<small>They define what an agent is allowed to access or do.  

Without boundaries, an agent could:  
- delete files,  
- send emails,  
- execute dangerous commands,  
- leak confidential data,  
- make unauthorized purchases.</nsmall>

**Principle of Least Privilege :**  
<small>An agent should only receive:    
- the minimum tools,  
- minimum data,  
- minimum permissions needed for the task.  

Example:  
A calendar scheduling agent:  
- Can read calendar events  
- Can create meetings  
- Cannot access banking apps  
- Cannot execute shell commands</nsmall>

<small>Tool Access Categories
| Tool Type                | Access Level   | Example                             |
| ------------------------ | -------------- | ----------------------------------- |
| Read-only tools          | Safer          | Search API, documentation retrieval |
| Write tools              | Medium risk    | Email sender, database update       |
| Execution tools          | High risk      | Shell commands, Python execution    |
| Financial/critical tools | Very high risk | Payment systems, cloud deployment   |</nsmall>

**Tool Whitelisting :**  
<small>Agents should use only explicitly approved tools.  

ALLOWED_TOOLS = [  
&emsp;"search_docs",  
&emsp;"calendar_api",  
&emsp;"weather_api"  
]  
if requested_tool not in ALLOWED_TOOLS:  
&emsp;raise PermissionError("Tool access denied")</nsmall>

**Sandboxing :**  
<small>Running agents in isolated environments like:  
- containers,  
- virtual machines,  
- restricted APIs,  
- temporary credentials.  

This prevents system-wide damage if the agent behaves unexpectedly.</nsmall>

#### Loop Limits and Runaway Prevention

**Timeout Limits**  
<small>Prevent excessive execution time.  

import time  
start = time.time()  
if time.time() - start > Threshold:  
&emsp;stop_agent("Execution timeout")</nsmall>

**Duplicate Action Detection**  
<small>Prevent repeating same failed action.  

Example:  
if current_action in previous_actions:  
&emsp;repeated_count += 1  
if repeated_count > threshold:  
&emsp;stop_agent("Repeated action detected")</nsmall>

#### Approval steps for high-stakes Actions

<small>High-impact actions should require human approval before execution.  
This is known as Human-in-the-loop (HITL), confirmation gating or approval workflows.  

Examples:  
Sending external emails  
Deploying production code  
Financial transactions  
Deleting databases  
Modifying legal documents  
Accessing private records</nsmall>

**Approval Workflow** 

<small>**Agent Plans Action**  
Agent wants to: "Delete 500 inactive customer records"  

**Summary**  
Proposed Action:   
- Delete 500 records  
- Database: customers  
- Reason: inactive > 2 years  

**Human Approval**  
Approve? (yes/no)  

**Execute Only After Approval**  
if human_approved:  
&emsp;execute_action()  
else:   
&emsp;cancel()</nsmall>

**Logging** 
<small> 
| Event                   | Example                        |
| ----------------------- | ------------------------------ |
| User request            | "Book a meeting tomorrow"      |
| Agent reasoning summary | "Need calendar access"         |
| Tool calls              | `calendar.create_event()`      |
| Tool outputs            | "Meeting created successfully" |
| Errors                  | "API timeout"                  |
| Approval requests       | "Awaiting user confirmation"   |
| Final actions           | "Email sent"                   |

Structured Logging Example:  
{
&ensp;"timestamp": "2026-05-18T10:30:00",  
&emsp;"agent": "scheduler_agent",  
&emsp;"step": 4,  
&emsp;"action": "create_calendar_event",  
&emsp;"status": "success"  
}

Do NOT log:  
- hidden chain-of-thought,  
- sensitive credentials,  
- personal secrets,  
- raw authentication tokens  

Instead log:  
- concise reasoning summaries  
- action intents  
- decision outcomes</nsmall>

**Tool Failure Scenario**  
<small>An agent tries to email a report.  
email_api.send() → SMTP timeout  

**Unsafe Behavior**  
- Retry forever  
- Retry every second  
- Spam the API  

**Escalation**  
After repeated failures:  
- notify administrator
- log the issue
- pause execution
- request human intervention.